In [1]:
!pip install -q scikit-survival

import numpy as np
import pandas as pd
import warnings
import os
import time as timer
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.util import Surv

warnings.filterwarnings('ignore')
np.random.seed(777)
HORIZONS_PRED = np.array([12, 24, 48, 72], dtype=float)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 19.3 MB/s eta 0:00:00


In [2]:
def locate_datasets():
    train_path, test_path = 'train.csv', 'test.csv'
    for search_root in ['/kaggle/input', '../input']:
        if os.path.exists(search_root):
            for root, _, files in os.walk(search_root):
                if 'train.csv' in files: train_path = os.path.join(root, 'train.csv')
                if 'test.csv' in files: test_path = os.path.join(root, 'test.csv')
    return train_path, test_path

train_path, test_path = locate_datasets()
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print(f'Training: {len(train_df)} samples, Test: {len(test_df)} samples')

Training: 221 samples, Test: 95 samples


In [3]:
def create_features(df):
    result = df.copy()
    dist = result['dist_min_ci_0_5h'].clip(lower=1)
    speed = result['closing_speed_m_per_h']
    perimeters = result['num_perimeters_0_5h']
    area_first = result['area_first_ha']
    result['log_distance'] = np.log1p(dist)
    result['inv_distance'] = 1 / (dist / 1000 + 0.1)
    result['inv_distance_sq'] = result['inv_distance'] ** 2
    result['sqrt_distance'] = np.sqrt(dist)
    result['dist_km'] = dist / 1000
    result['dist_km_sq'] = (dist / 1000) ** 2
    result['dist_rank'] = dist.rank(pct=True)
    fire_radius = np.sqrt(area_first * 10000 / np.pi)
    result['radius_to_dist'] = fire_radius / dist
    result['area_to_dist_ratio'] = area_first / (dist / 1000 + 0.1)
    result['log_area_dist_ratio'] = np.log1p(area_first) - np.log1p(dist)
    result['has_movement'] = (perimeters > 1).astype(float)
    closing_pos = speed.clip(lower=0)
    result['eta_hours'] = np.where(closing_pos > 0.01, dist / closing_pos, 9999).clip(max=9999)
    result['log_eta'] = np.log1p(result['eta_hours'].clip(0, 9999))
    radial_growth = result['radial_growth_rate_m_per_h'].clip(lower=0)
    effective_closing = closing_pos + radial_growth
    result['effective_closing_speed'] = effective_closing
    result['eta_effective'] = np.where(effective_closing > 0.01, dist / effective_closing, 9999).clip(max=9999)
    result['threat_score'] = result['alignment_abs'] * speed / np.log1p(dist)
    result['fire_urgency'] = perimeters * speed
    result['growth_intensity'] = result['area_growth_rate_ha_per_h'] * perimeters
    result['zone_critical'] = (dist < 5000).astype(float)
    result['zone_warning'] = ((dist >= 5000) & (dist < 10000)).astype(float)
    result['zone_safe'] = (dist >= 10000).astype(float)
    result['is_summer'] = result['event_start_month'].isin([6, 7, 8]).astype(float)
    result['is_afternoon'] = ((result['event_start_hour'] >= 12) & (result['event_start_hour'] < 20)).astype(float)
    drop_cols = ['relative_growth_0_5h', 'projected_advance_m', 'centroid_displacement_m',
                 'centroid_speed_m_per_h', 'closing_speed_abs_m_per_h', 'area_growth_abs_0_5h']
    result = result.drop(columns=[c for c in drop_cols if c in result.columns])
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)
    return result

train_processed = create_features(train_df)
test_processed = create_features(test_df)

In [4]:
def get_surv_predictions(model, X):
    surv_fns = model.predict_survival_function(X)
    preds = np.empty((len(surv_fns), len(HORIZONS_PRED)), dtype=float)
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        preds[i, :] = fn(np.clip(HORIZONS_PRED, t_min, t_max))
    return 1.0 - preds

def sigmoid_pred(dist, threshold, scale):
    return 1.0 / (1.0 + np.exp((dist - threshold) / scale))

def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(float)
    return y, ~unknown

def compute_ipcw_weights(times, events, horizon):
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t))
    for i, t in enumerate(unique_t):
        at_risk = (times >= t).sum()
        censored_at_t = ((times == t) & (events == 0)).sum()
        if at_risk > 0: surv[i] = 1 - censored_at_t / at_risk
        if i > 0: surv[i] *= surv[i - 1]
    def G(t):
        idx = np.searchsorted(unique_t, t, side='right') - 1
        return max(surv[idx], 0.01) if idx >= 0 else 1.0
    weights = np.ones(len(times))
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon: weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon: weights[i] = 1.0 / G(horizon)
    return weights

def enforce_monotonicity(preds):
    result = np.clip(preds, 0, 1)
    for i in range(1, result.shape[1]):
        result[:, i] = np.maximum(result[:, i], result[:, i-1])
    return result

In [5]:
X_surv_train = train_df.drop(columns=['event_id', 'event', 'time_to_hit_hours'])
X_surv_test = test_df.drop(columns=['event_id'])
y_surv = Surv.from_arrays(event=train_df['event'].astype(bool), time=train_df['time_to_hit_hours'])
event_values = train_df['event'].values
time_values = train_df['time_to_hit_hours'].values
dist_test = test_df['dist_min_ci_0_5h'].values

gbsa_configs = [
    {'learning_rate': 0.01, 'subsample': 0.7,  'max_depth': 3, 'min_samples_leaf': 12, 'min_samples_split': 3, 'n_estimators': 1200, 'dropout_rate': 0.0},
    {'learning_rate': 0.01, 'subsample': 0.85, 'max_depth': 3, 'min_samples_leaf': 15, 'min_samples_split': 3, 'n_estimators': 1200, 'dropout_rate': 0.0},
    {'learning_rate': 0.01, 'subsample': 0.6,  'max_depth': 3, 'min_samples_leaf': 12, 'min_samples_split': 3, 'n_estimators': 1200, 'dropout_rate': 0.0},
    {'learning_rate': 0.005,'subsample': 0.85, 'max_depth': 3, 'min_samples_leaf': 12, 'min_samples_split': 3, 'n_estimators': 2000, 'dropout_rate': 0.0},
    {'learning_rate': 0.01, 'subsample': 0.85, 'max_depth': 3, 'min_samples_leaf': 20, 'min_samples_split': 3, 'n_estimators': 1400, 'dropout_rate': 0.0},
]
SEEDS = (123, 456, 789, 777, 666,
         1511, 1523, 2025, 2026, 2033,
        279, 239, 70, 77, 31,
        2024, 2077, 3077, 123456, 654321)
N_SEEDS = len(SEEDS)

test_gbsa = np.zeros((len(X_surv_test), 4))
total_models = len(gbsa_configs) * N_SEEDS * 5
model_count = 0
t_start = timer.time()

for cfg_idx, cfg in enumerate(gbsa_configs):
    cfg_test = np.zeros((len(X_surv_test), 4))
    for seed in SEEDS:
        seed_test = np.zeros((len(X_surv_test), 4))
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for fold_idx, (tr_idx, va_idx) in enumerate(cv.split(X_surv_train, event_values)):
            m = GradientBoostingSurvivalAnalysis(**{**cfg, 'random_state': seed})
            m.fit(X_surv_train.iloc[tr_idx], y_surv[tr_idx])
            seed_test += get_surv_predictions(m, X_surv_test) / 5
            model_count += 1
        cfg_test += seed_test / N_SEEDS
    test_gbsa += cfg_test / len(gbsa_configs)
    elapsed = timer.time() - t_start
    print(f'Config {cfg_idx+1}/{len(gbsa_configs)} done [{model_count}/{total_models}, {elapsed/60:.1f}m]')

# PowerCal 24h
test_gbsa[:, 1] = np.clip(test_gbsa[:, 1] ** 1.1, 0, 1)
print(f'GBSA done: {total_models} fold models')

Config 1/5 done [100/500, 2.2m]
Config 2/5 done [200/500, 4.6m]
Config 3/5 done [300/500, 6.7m]
Config 4/5 done [400/500, 10.6m]
Config 5/5 done [500/500, 13.3m]
GBSA done: 500 fold models


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# =========================================================
# Best validated path:
# Global LGB heads + sparse-near subgroup heads/meta
# PLUS new sparse-near KNN analog timing branch
# =========================================================

X_lgb_train = train_processed.drop(columns=['event_id', 'event', 'time_to_hit_hours'])
X_lgb_test = test_processed.drop(columns=['event_id'])

lgb_cfgs = {
    24: {'max_depth': 3, 'learning_rate': 0.03, 'n_estimators': 300,
         'subsample': 0.7, 'colsample_bytree': 0.7, 'min_child_samples': 8,
         'reg_alpha': 0.5, 'reg_lambda': 2.0, 'num_leaves': 7},
    48: {'max_depth': 2, 'learning_rate': 0.05, 'n_estimators': 200,
         'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_samples': 5,
         'reg_alpha': 0.1, 'reg_lambda': 1.0, 'num_leaves': 4},
}

LGB_SEEDS = (123, 456, 789, 777, 666,
             1511, 1523, 2025, 2026, 2033,
             279, 239, 70, 77, 31,
             2024, 2077, 3077, 123456, 654321,
             2034, 2035, 2036, 1984, 1991)

lgb_test = {}
lgb_oof = {}

for horizon in [24, 48]:
    y_bin, mask = make_binary_target(time_values, event_values, horizon)
    valid_idx = np.where(mask)[0]

    pos = int(y_bin[mask].sum())
    neg = int(mask.sum() - pos)
    n_splits = max(2, min(5, pos, neg))

    cfg = lgb_cfgs[horizon]
    oof_sum = np.zeros(len(X_lgb_train))
    oof_cnt = np.zeros(len(X_lgb_train))
    test_sum = np.zeros(len(X_lgb_test))

    for seed in LGB_SEEDS:
        seed_test = np.zeros(len(X_lgb_test))
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

        for tr_v, va_v in cv.split(valid_idx, y_bin[mask]):
            tr_idx = valid_idx[tr_v]
            va_idx = valid_idx[va_v]

            ipcw_w = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)

            m = lgb.LGBMClassifier(
                **cfg,
                objective='binary',
                random_state=seed,
                verbose=-1
            )
            m.fit(X_lgb_train.iloc[tr_idx], y_bin[tr_idx], sample_weight=ipcw_w)

            pv = m.predict_proba(X_lgb_train.iloc[va_idx])[:, 1]
            pt = m.predict_proba(X_lgb_test)[:, 1]

            oof_sum[va_idx] += pv
            oof_cnt[va_idx] += 1
            seed_test += pt / n_splits

        test_sum += seed_test / len(LGB_SEEDS)

    oof = np.divide(oof_sum, np.maximum(oof_cnt, 1))
    fill_val = float(y_bin[mask].mean())
    oof[oof_cnt == 0] = fill_val

    lgb_oof[horizon] = np.clip(oof, 0, 1)
    lgb_test[horizon] = np.clip(test_sum, 0, 1)

    print(f'LGB {horizon}h done | valid={int(mask.sum())} | pos={pos} | neg={neg} | folds={n_splits}')

# ---------------------------------------------------------
# Masks
# ---------------------------------------------------------
dist_train = train_df['dist_min_ci_0_5h'].values
dist_test = test_df['dist_min_ci_0_5h'].values
lowtemp_train = train_df['low_temporal_resolution_0_5h'].astype(int).values
lowtemp_test = test_df['low_temporal_resolution_0_5h'].astype(int).values

near_train = dist_train < 5000
near_test = dist_test < 5000
far_test = ~near_test

stable_train = near_train & (lowtemp_train == 0)
stable_test = near_test & (lowtemp_test == 0)

sparse_train = near_train & (lowtemp_train == 1)
sparse_test = near_test & (lowtemp_test == 1)

# ---------------------------------------------------------
# Sparse-near LR heads
# ---------------------------------------------------------
SPARSE_FEATURES = [
    'dt_first_last_0_5h',
    'num_perimeters_0_5h',
    'eta_effective',
    'alignment_abs',
    'log_area_dist_ratio',
    'effective_closing_speed',
    'log_distance',
    'fire_urgency',
    'inv_distance',
]

X_sparse_train = train_processed.loc[sparse_train, SPARSE_FEATURES].reset_index(drop=True)
X_sparse_test = test_processed.loc[sparse_test, SPARSE_FEATURES].reset_index(drop=True)
time_sparse = train_df.loc[sparse_train, 'time_to_hit_hours'].values

SPARSE_LR_SEEDS = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033)
sparse_lr_oof = {}
sparse_lr_test = {}

for horizon, C in [(12, 0.70), (24, 0.40), (48, 0.25)]:
    y_h = (time_sparse <= horizon).astype(int)
    counts = np.bincount(y_h, minlength=2)
    n_splits = max(2, min(5, int(counts.min())))

    oof_sum = np.zeros(len(X_sparse_train))
    oof_cnt = np.zeros(len(X_sparse_train))
    test_sum = np.zeros(len(X_sparse_test))

    for seed in SPARSE_LR_SEEDS:
        seed_test = np.zeros(len(X_sparse_test))
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

        for tr_idx, va_idx in cv.split(X_sparse_train, y_h):
            m = Pipeline([
                ('sc', StandardScaler()),
                ('lr', LogisticRegression(
                    C=C,
                    max_iter=5000,
                    solver='lbfgs',
                    class_weight='balanced'
                ))
            ])
            m.fit(X_sparse_train.iloc[tr_idx], y_h[tr_idx])

            pv = m.predict_proba(X_sparse_train.iloc[va_idx])[:, 1]
            pt = m.predict_proba(X_sparse_test)[:, 1]

            oof_sum[va_idx] += pv
            oof_cnt[va_idx] += 1
            seed_test += pt / n_splits

        test_sum += seed_test / len(SPARSE_LR_SEEDS)

    oof = np.divide(oof_sum, np.maximum(oof_cnt, 1))
    fill_val = float(y_h.mean())
    oof[oof_cnt == 0] = fill_val

    sparse_lr_oof[horizon] = np.clip(oof, 0, 1)
    sparse_lr_test[horizon] = np.clip(test_sum, 0, 1)

    print(f'SPARSE LR {horizon}h done | n={len(y_h)} | pos={int(y_h.sum())} | neg={int((1-y_h).sum())} | folds={n_splits}')

# ---------------------------------------------------------
# Sparse-near meta heads
# ---------------------------------------------------------
SPARSE_META_FEATURES = [
    'dt_first_last_0_5h',
    'num_perimeters_0_5h',
    'eta_effective',
    'alignment_abs',
    'log_area_dist_ratio',
    'effective_closing_speed',
    'log_distance',
    'inv_distance',
]

META_SEEDS = (101, 202, 303, 404, 505, 606, 707, 808, 909, 1001)
sparse_meta_test = {}

for horizon, C in [(12, 0.55), (24, 0.32), (48, 0.22)]:
    y_h = (time_sparse <= horizon).astype(int)

    base_train_cols = [sparse_lr_oof[horizon]]
    base_test_cols = [sparse_lr_test[horizon]]

    if horizon in (24, 48):
        base_train_cols.append(lgb_oof[horizon][sparse_train])
        base_test_cols.append(lgb_test[horizon][sparse_test])

    raw_train = train_processed.loc[sparse_train, SPARSE_META_FEATURES].values
    raw_test = test_processed.loc[sparse_test, SPARSE_META_FEATURES].values

    M_train = np.concatenate([np.column_stack(base_train_cols), raw_train], axis=1)
    M_test = np.concatenate([np.column_stack(base_test_cols), raw_test], axis=1)

    counts = np.bincount(y_h, minlength=2)
    n_splits = max(2, min(5, int(counts.min())))

    test_sum = np.zeros(len(M_test))

    for seed in META_SEEDS:
        seed_test = np.zeros(len(M_test))
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

        for tr_idx, va_idx in cv.split(M_train, y_h):
            m = Pipeline([
                ('sc', StandardScaler()),
                ('lr', LogisticRegression(
                    C=C,
                    max_iter=5000,
                    solver='lbfgs',
                    class_weight='balanced'
                ))
            ])
            m.fit(M_train[tr_idx], y_h[tr_idx])
            seed_test += m.predict_proba(M_test)[:, 1] / n_splits

        test_sum += seed_test / len(META_SEEDS)

    sparse_meta_test[horizon] = np.clip(test_sum, 0, 1)
    print(f'SPARSE META {horizon}h done | folds={n_splits}')

# ---------------------------------------------------------
# NEW: sparse-near KNN analog timing branch
# This is the only new branch vs 0.97503
# ---------------------------------------------------------
KNN_FEATURES = [
    'log_area_dist_ratio',
    'dist_min_ci_0_5h',
    'event_start_hour',
    'event_start_month',
    'inv_distance',
    'log_distance',
]

X_knn_train = train_processed.loc[sparse_train, KNN_FEATURES].reset_index(drop=True)
X_knn_test = test_processed.loc[sparse_test, KNN_FEATURES].reset_index(drop=True)
time_knn = train_df.loc[sparse_train, 'time_to_hit_hours'].values

sparse_knn_test = {12: np.zeros(len(X_knn_test)), 24: np.zeros(len(X_knn_test)), 48: np.zeros(len(X_knn_test))}

if len(X_knn_test) > 0:
    scaler = StandardScaler()
    Z_train = scaler.fit_transform(X_knn_train)
    Z_test = scaler.transform(X_knn_test)

    K_LIST = [5, 7, 9, 11, 15]
    K_LIST = [k for k in K_LIST if k <= len(Z_train)]

    for k in K_LIST:
        nn = NearestNeighbors(n_neighbors=k, metric='euclidean')
        nn.fit(Z_train)
        dists, inds = nn.kneighbors(Z_test)

        weights = 1.0 / (dists + 0.15)
        weights = weights / weights.sum(axis=1, keepdims=True)

        for horizon in [12, 24, 48]:
            y_h = (time_knn <= horizon).astype(float)
            pred_h = (weights * y_h[inds]).sum(axis=1)
            sparse_knn_test[horizon] += pred_h / len(K_LIST)

for horizon in [12, 24, 48]:
    sparse_knn_test[horizon] = np.clip(sparse_knn_test[horizon], 0, 1)
    print(f'SPARSE KNN {horizon}h mean={sparse_knn_test[horizon].mean() if len(X_knn_test)>0 else 0:.4f}')


LGB 24h done | valid=196 | pos=63 | neg=133 | folds=5
LGB 48h done | valid=166 | pos=66 | neg=100 | folds=5
SPARSE LR 12h done | n=33 | pos=15 | neg=18 | folds=5
SPARSE LR 24h done | n=33 | pos=28 | neg=5 | folds=5
SPARSE LR 48h done | n=33 | pos=30 | neg=3 | folds=3
SPARSE META 12h done | folds=5
SPARSE META 24h done | folds=5
SPARSE META 48h done | folds=3
SPARSE KNN 12h mean=0.4827
SPARSE KNN 24h mean=0.8892
SPARSE KNN 48h mean=0.9517


In [7]:
# =========================================================
# Final blend: 0.97503 winner + light sparse-KNN analog correction
# =========================================================

W24 = 0.95
W48 = 0.45

test_blend = test_gbsa.copy()

# strong global anchor
test_blend[:, 1] = W24 * test_gbsa[:, 1] + (1 - W24) * lgb_test[24]
test_blend[:, 2] = W48 * test_gbsa[:, 2] + (1 - W48) * lgb_test[48]

def rate_within(mask, horizon):
    return float((train_df.loc[mask, 'time_to_hit_hours'] <= horizon).mean())

p12_stable = rate_within(stable_train, 12)
p24_stable = rate_within(stable_train, 24)
p48_stable = rate_within(stable_train, 48)

# 1) Far fires: structural zero
test_blend[far_test, :] = 0.0

# 2) Stable near fires: keep proven 0.97503 rails
test_blend[stable_test, 0] = np.maximum(
    0.78 * test_blend[stable_test, 0] + 0.22 * p12_stable,
    0.90
)
test_blend[stable_test, 1] = np.maximum(
    0.75 * test_blend[stable_test, 1] + 0.25 * p24_stable,
    0.965
)
test_blend[stable_test, 2] = np.maximum(
    0.80 * test_blend[stable_test, 2] + 0.20 * p48_stable,
    0.995
)

# 3) Sparse near fires: proven sparse-specialized model
if sparse_test.sum() > 0:
    test_blend[sparse_test, 0] = (
        0.58 * test_blend[sparse_test, 0] + 0.42 * sparse_meta_test[12]
    )
    test_blend[sparse_test, 1] = (
        0.70 * test_blend[sparse_test, 1] + 0.30 * sparse_meta_test[24]
    )
    test_blend[sparse_test, 2] = (
        0.62 * test_blend[sparse_test, 2] + 0.38 * sparse_meta_test[48]
    )

    # original sparse LR stabilizer from 0.97503
    test_blend[sparse_test, 0] = (
        0.92 * test_blend[sparse_test, 0] + 0.08 * sparse_lr_test[12]
    )
    test_blend[sparse_test, 1] = (
        0.95 * test_blend[sparse_test, 1] + 0.05 * sparse_lr_test[24]
    )
    test_blend[sparse_test, 2] = (
        0.95 * test_blend[sparse_test, 2] + 0.05 * sparse_lr_test[48]
    )

    # NEW: light KNN analog correction
    # Small enough not to destroy 0.97503, but enough to help if analog timing is right.
    test_blend[sparse_test, 0] = (
        0.90 * test_blend[sparse_test, 0] + 0.10 * sparse_knn_test[12]
    )
    test_blend[sparse_test, 1] = (
        0.90 * test_blend[sparse_test, 1] + 0.10 * sparse_knn_test[24]
    )
    test_blend[sparse_test, 2] = (
        0.84 * test_blend[sparse_test, 2] + 0.16 * sparse_knn_test[48]
    )

    # anti-overfit guard from failed 0.96904:
    # never crush sparse 48h too low; 48h is heavily weighted.
    test_blend[sparse_test, 2] = np.maximum(test_blend[sparse_test, 2], 0.88)

# deterministic 72h under train support
test_blend[near_test, 3] = 1.0
test_blend[far_test, 3] = 0.0

test_final = enforce_monotonicity(test_blend)

submission = pd.DataFrame({
    'event_id': test_df['event_id'],
    'prob_12h': test_final[:, 0],
    'prob_24h': test_final[:, 1],
    'prob_48h': test_final[:, 2],
    'prob_72h': test_final[:, 3],
})

output_path = '/kaggle/working/submission.csv' if os.path.isdir('/kaggle/working') else 'submission.csv'
submission.to_csv(output_path, index=False)

print("stable_train_count =", int(stable_train.sum()))
print("sparse_train_count =", int(sparse_train.sum()))
print("stable_test_count =", int(stable_test.sum()))
print("sparse_test_count =", int(sparse_test.sum()))
print("p12_stable =", round(float(p12_stable), 6))
print("p24_stable =", round(float(p24_stable), 6))
print("p48_stable =", round(float(p48_stable), 6))
print("sparse_knn_12_mean =", round(float(np.mean(sparse_knn_test[12])) if sparse_test.sum() else 0, 6))
print("sparse_knn_24_mean =", round(float(np.mean(sparse_knn_test[24])) if sparse_test.sum() else 0, 6))
print("sparse_knn_48_mean =", round(float(np.mean(sparse_knn_test[48])) if sparse_test.sum() else 0, 6))
print(f"Saved: {output_path}")

submission.describe().round(4)

stable_train_count = 36
sparse_train_count = 33
stable_test_count = 10
sparse_test_count = 18
p12_stable = 0.944444
p24_stable = 0.972222
p48_stable = 1.0
sparse_knn_12_mean = 0.482666
sparse_knn_24_mean = 0.889232
sparse_knn_48_mean = 0.951655
Saved: /kaggle/working/submission.csv


,event_id,prob_12h,prob_24h,prob_48h,prob_72h
count,9.500000e+01,95.0000,95.0000,95.0000,95.0000
mean,5.695393e+07,0.1949,0.2491,0.2783,0.2947
std,2.329721e+07,0.3292,0.3932,0.4335,0.4583
min,1.066260e+07,0.0000,0.0000,0.0000,0.0000
25%,4.027536e+07,0.0000,0.0000,0.0000,0.0000
50%,5.480111e+07,0.0000,0.0000,0.0000,0.0000
75%,7.536942e+07,0.3889,0.7302,0.8800,1.0000
max,9.964946e+07,0.9870,0.9923,0.9976,1.0000
